**This is the Obsidian Section**


>Imports


In [ ]:
# Standard Library
import json
import os
import re
import sys
import time
import warnings
from collections import defaultdict
from typing import Any, Dict, List, Set, Tuple
import yaml
from pathlib import Path
# Third-Party Libraries
import matplotlib.pyplot as plt
import nltk
import numpy as np
import ollama
import pandas as pd
import psycopg2
import requests
import torch
from huggingface_hub import login
from langdetect import detect
from nltk.tokenize import sent_tokenize, word_tokenize
from scipy.stats import linregress
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sqlalchemy import create_engine

# Local Application Imports
from diary_analyzer.dataframe_processing import (
    check_coocurrence,
    convert_to_list,
    generate_specialized_dataframe_with_counts,
    get_all_values_of_column,
    preprocess_dataframe,
    replace_and_flatten_activities,
    solve_missing_separation,
    correct_permutations,
)
from diary_analyzer.embedding import classify_dataframe
from diary_analyzer.latex import generate_latex_report
from diary_analyzer.ollama_stuff import check_name_typos, parse_name_groups, strip_reasoning
from diary_analyzer.parse_obsidian import iterate_files
from diary_analyzer.spelling import language_spelling_cleanup
from diary_analyzer.yaml_functions import load_config_file, get_classification_data
from diary_analyzer.visualization_personal import (
    plot_boxplot_of_weekdays,
    visualize_rating_progression,
    plot_sleep_duration,
    save_fig
)

# parameters
use_old = True # to avoid recomputation use the existing database if calculations have been done before

In [ ]:
# some of the calculations are more extensive to not recompute everything all over again we keep the option

use_old = True
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. Load the variables from the .env file
load_dotenv()
def load_old_entries():

    # 2. Retrieve them using os.getenv
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    DB_HOST = "localhost" 
    DB_PORT = os.getenv("DB_PORT")
    DB_NAME = os.getenv("DB_NAME")
 

    # 3. Build the connection string
    connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{'daily_notes'}"
    engine = create_engine(connection_string)

    df = pd.read_sql("SELECT * FROM daily_notes", engine)
    return df
if use_old == True:
    old_df = load_old_entries()


# use_old = False  #uncomment for starting from scratch 

>config

In [ ]:


# Get the path to the root directory (one level up from 'notebooks')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to the system path
if project_root not in sys.path:
    sys.path.append(project_root)



# Folder containing Markdown files
folder_path ="../data"
# Create DataFrame

In [ ]:
# visualize the dataframe
df = iterate_files(folder_path)

# Display the DataFrame
print(df)


In [ ]:





df_sorted = preprocess_dataframe(df)


#df_sorted = convert_to_list(df_sorted, 'People')
#df = convert_to_list(df_sorted, 'Activities')



In [ ]:
print(old_df.shape)
print(old_df.columns)
print(df_sorted.shape)
print(df_sorted.columns)
print(set(df_sorted.columns) - set(old_df.columns))
df_sorted.set_index(['Date', 'content'], inplace=True)
old_df.set_index(['Date', 'content'], inplace=True)

# 2. This tells pandas: "Keep df_sorted values, but if they are NaN, 
# pull the value from old_df"
combined_df = df_sorted.combine_first(old_df).reset_index()

In [ ]:
df_sorted = combined_df

In [ ]:

df = plot_sleep_duration(df)


In [ ]:

df = language_spelling_cleanup(df_sorted)



In [ ]:
# dataframe preprocessing 
# filter out the empty entries
#df_sorted.shape
#df = df_sorted
print(df[df['language'] == 'fr'].shape)
# filtered out 1 line here

In [ ]:


df = correct_permutations(df, "People")

In [ ]:

# Example usage with your data
if __name__ == "__main__":
    # Example name list with various issues
    test_names = [
        "John Smith", "Jon Smith", "Jane Doe", "Jahn Smith", 
        "Jane Dow", "Michael Johnson", "Mike Johnson", "sister",
        "tom tailer", "Tom Taylor", "Alex", "Alex(online)",
        "Sarah Connor", "Sara Connor", "Bob Wilson", "Robert Wilson"
    ]
    
    # Custom rules for your specific dataset for example if names are inconsistent here complex rules can be defined.
    custom_rules = os.getenv("name_correction_prompts")
    
    # Process names
    result = check_name_typos(all_people, custom_rules=custom_rules)
    print(result)
    clean_result = strip_reasoning(result)
    parsed_groups = parse_name_groups(clean_result)
    
    print("Raw model output:")
    print(result)
    print("\nCleaned output:")
    print(clean_result)
    print("\nParsed groups:")
    for i, group in enumerate(parsed_groups, 1):
        print(f"Group {i}: {group}")

>Parsing of the data

In [ ]:
# Activity analysis 


# Finden von Schreibfehlern

# Gesamtliste





In [ ]:
# Check what dart co-occurs with
check_coocurrence(df, 'Activities3', 'Frisbee')
df.shape

In [ ]:

df["Essen"].dtype
print(df["Essen"].value_counts())
essen_df, all_essen = get_all_values_of_column(df, "Essen")
trinken_df, all_trinken = get_all_values_of_column(df, "Trinken")
print(sorted(list(set(all_essen))))

In [ ]:

essen_df = generate_specialized_dataframe_with_counts(essen_df, 'Essen3')

trinken_df = generate_specialized_dataframe_with_counts(trinken_df, 'Trinken3')
print(essen_df.head(10))
print(trinken_df.head(10))


# Apply to a DataFrame where each entry is a list of items


In [ ]:

# --- CORE NORMALIZATION LOGIC ---

def _process_item_core(item):
    """
    Core logic to extract quantity, convert to liters, and clean the item name.
    
    Returns (standardized_name, quantity_in_liters)
    """
    item_str = str(item).strip()

    # 1. Regex to capture: (Quantity) (Optional Unit/Multiplier) (Item Name)
    # The regex targets standard quantity formats and common liquid units (L, ML, etc.).
    # Quantity: (\d*\.?\d+) 
    # Unit: (\s*[a-zA-Z]*) -> Optional space, then optional letters (for unit)
    # Item: (.*) -> Everything else is the item name (or multiplier 'x')
    match = re.match(r'^(\d*\.?\d+)\s*([a-zA-Z]*)\s*(.*)$', item_str, re.IGNORECASE)

    quantity = 1.0 # Default if no number is found
    unit = ""
    raw_name = item_str
    
    if match:
        quantity_str, unit, raw_name = match.groups()
        try:
            quantity = float(quantity_str)
        except ValueError:
             # Fallback if float conversion fails
             quantity = 1.0
             raw_name = item_str
    
    # --- 2. Standardize Unit and Quantity (Conversion to Liters) ---
    unit = unit.lower().strip()
    
    quantity_in_liters = quantity
    
    if unit in ('ml'):
        # Convert milliliters to liters
        quantity_in_liters /= 1000.0
    elif unit in ('x', '', 'stk'):
        # If the unit is 'x' (multiplier) or missing, we assume the item 
        # is a standard serving size (e.g., one can of Cola).
        # We need a defined standard size for aggregation. Let's assume 
        # a standard serving is 0.33 Liters (e.g., a small bottle/can)
        # You may need to adjust this factor based on your data!
        quantity_in_liters *= 0.33
    
    # --- 3. Clean Item Name ---
    temp_name = raw_name.strip()
    
    # Strip common prefixes/units from the name if they were left there
    unit_prefixes = ['g', 'l', 'ml', 'gr', 'lt', 'x', 'stk']
    
    for prefix in unit_prefixes:
        if temp_name.lower().startswith(prefix):
            temp_name = temp_name[len(prefix):].strip()
            continue
            
    # Final Standardization
    standardized_name = temp_name.strip().title()

    # --- 4. Final Aggregation Mapping (Unification) ---
    # Manually map common variations to a single, consistent name.
    if "Wasser" in standardized_name:
        standardized_name = "Water"
    elif "Bier" in standardized_name or "Weizen" in standardized_name or "Kilkenny" in standardized_name or "Radler" in standardized_name:
        standardized_name = "Beer/Malt"
    elif "Cola" in standardized_name or "Mate" in standardized_name:
        standardized_name = "Soda/Mate"
    elif "Apfelschorle" in standardized_name:
        standardized_name = "Apfelschorle"
    elif not standardized_name and quantity_in_liters > 0:
        # If the name is blank but we have a quantity, assign it to a category
        standardized_name = "Uncategorized Drink"

    # Skip items where the name is still empty (e.g., rows that were just empty delimiters)
    if not standardized_name:
        return "", 0.0
        
    return standardized_name, quantity_in_liters


# --- COMPATIBILITY WRAPPERS (Unchanged) ---

def normalize_item(item):
    """Placeholder for backward compatibility. Uses core logic."""
    name, quantity = _process_item_core(item)
    return f"{quantity} {name}" if name else str(item).strip()

def normalize_and_count(item):
    """Extract item name and quantity (default to 1 if no prefix). Uses core logic."""
    return _process_item_core(item)


# --- REVISED MAIN PROCESSING FUNCTION (Crucially, handles string splitting) ---

def generate_specialized_dataframe_with_counts(df, column_name):
    """
    Processes the specified DataFrame column to calculate total item counts 
    after robust extraction and unit conversion.
    """
    
    item_counts = defaultdict(float)

    # Directly iterate over the column containing the raw lists
    for raw_entry in df[column_name]:
        
        # 1. Skip NaN/Null rows
        if pd.isna(raw_entry):
            continue
            
        # 2. Extract individual items (handling list format OR single string format)
        if isinstance(raw_entry, list):
            items_to_process = raw_entry
        elif isinstance(raw_entry, str):
            # Handles the packed string format from value_counts, e.g., 
            # "[0.5LWeizen, 1LApfelschorle, 2LWasser]"
            # Strip brackets, then split by comma, and strip spaces
            items_to_process = [
                item.strip() for item in raw_entry.strip('[]').split(',') if item.strip()
            ]
        else:
            # Handle unexpected types (e.g., single floats from errors)
            continue
            
        # 3. Process each individual item
        for item in items_to_process:
            
            # Use the core normalization logic directly on the RAW item string
            name, quantity = _process_item_core(item)
            
            # Aggregate counts, skipping empty names
            if name:
                item_counts[name] += quantity

    # 4. Convert to a clean DataFrame for output
    count_df = pd.DataFrame(
        {"Item": item_counts.keys(), "Total Liters": item_counts.values()}
    ).sort_values("Total Liters", ascending=False).reset_index(drop=True)
    
    return count_df

trinken_df = generate_specialized_dataframe_with_counts(df, 'Trinken')
print(essen_df.head(10))

In [ ]:

config = load_config_file()
food_categories, food_prompts  = get_classification_data("food")

model_id = "google/embeddinggemma-300m"
# This downloads it and adds the necessary pooling layers
model_path = "/models/gemma-300m"
# Use the path inside the container that we set up in docker-compose.yml
model_id = "/root/.cache/huggingface/hub/models--google--embeddinggemma-300m/snapshots/57c266a740f537b4dc058e1b0cda161fd15afa75" # local path for docker after download
model = SentenceTransformer(model_id, trust_remote_code=True, local_files_only=True)

# Save the full, ready-to-use SentenceTransformer version
model.save("./my_local_gemma")

essen_df_classified = classify_dataframe(df=essen_df, classification_prompts=food_prompts, categories=food_categories)
drinks_categories, drinks_prompts  = get_classification_data("drinks")

trinken_df_classified = classify_dataframe(df=trinken_df, classification_prompts=drinks_prompts, categories=drinks_categories)



In [ ]:
print(essen_df_classified)
print(trinken_df_classified)

In [ ]:
mask = df['Trinken'].astype(str).str.lower().str.contains("weizen", na=False)
matching_indices = df[mask].index.tolist()
print(len(df[mask].index.tolist()))
print(f"Rows with 'bier': {matching_indices}")
print(df[mask]["Trinken"])

In [ ]:


df['date'] = pd.to_datetime(df['filename'].str.extract(r'(\d{2}-\d{2}-\d{4})')[0], format='%d-%m-%Y')
df_sorted = df.sort_values('date')
visualize_rating_progression(df_sorted)

In [ ]:
df['weekday'] = df['date'].dt.day_name()
df = df[df['date'] > '2025-01-01']
df = df[df['content'] != ""]
plot_boxplot_of_weekdays(df)


In [ ]:
def get_average_performance(df, column_name, total_list):
    average_dict = {x: (0,0) for x in total_list}  
    average_rating = df['rating'].mean()
    for index, row in df.iterrows():
        items = row[column_name]
        rating = row['rating']
        if isinstance(items, list):
            for item in items:
                item = item.lower()
                if item in average_dict:
                    current_sum, count = average_dict[item]
                    average_dict[item] = (current_sum + rating, count + 1)
    average_performance = {k: (v[0] / v[1] if v[1] > 0 else 0, v[1]) for k, v in average_dict.items()}
    sorted_average = [(k, v) for k, v in sorted(average_performance.items(), key=lambda item: item[1][0])]
    delta_performance = [(k, (v[0] - average_rating, v[1])) for k, v in sorted_average]
    return delta_performance
print(get_average_performance(df, 'Activities3', all_activities))

In [ ]:
all_activities
# check whether the activities contain substrings

df = solve_missing_separation(df, "Activities3", all_activities)

# --- Example Usage (Assuming you have a full, sorted list) ---

# Example data simulation (must be sorted!)
# all_activities_example = sorted([
#     "read", "reading", "readbook", "run", "running", "soccer", 
#     "soccerpractice", "practice", "work", "workout", "out"
# ])
# print(f"Example activities: {all_activities_example}")



# Expected Output Example: 
# {'readbook': ['read', 'book'], 
#  'running': ['run', 'ning'], # May occur if 'ning' is in the list
#  'soccerpractice': ['soccer', 'practice']}

> Generating the latex formation for printing

In [ ]:

# Generate LaTeX report

# Generate report
generate_latex_report(df)


In [ ]:
# save the content to the database
df.to_sql('daily_journal', engine, if_exists='replace', index=False)


In [ ]:
df = df[df['language'] == 'fr']
df = df[df['rating'] >= 8.6]
df

In [ ]:
import json
from typing import Any, Dict, List, Optional

import ollama
import pandas as pd
from pydantic import BaseModel, Field
from tqdm import tqdm  # Optional: for progress bar


# --- 1. Define the Desired Output Structure (Same as before) ---
class DiarySection(BaseModel):
    """Represents a single, self-contained, high-level topic or note from a daily diary entry."""
    section_title: str = Field(..., description="A short, descriptive title for this section.")
    topic_summary: str = Field(..., description="A complete, self-contained summary paragraph of the note's content. This will be the text used for embedding.")
    keywords: List[str] = Field(..., description="A list of 3-5 core keywords from this section to aid search.")

# Since we handle the date in the DataFrame, we don't need the outer DailySummary model 
# in the LLM output. We ask the LLM to return a list of sections directly.
class SectionsList(BaseModel):
    sections: List[DiarySection] = Field(..., description="A list of distinct, topically separate notes generated from the diary entry.")


def process_entry_for_rag(text_content: str, model_name: str = 'deepseek-r1:7b') -> List[Dict[str, Any]]:
    """
    Calls Ollama to generate structured, semantic chunks for a single diary entry.

    Returns: A list of dictionaries, where each dict is a ready-to-embed chunk.
    """
    
    # Define the output schema for the LLM
    output_schema = SectionsList.model_json_schema()

    # 1. Construct the System Prompt
    '''system_prompt = (
    "You are an expert diary analyst and a meticulous data extraction agent. "
    "Your core task is to process a single daily entry and perform **semantic chunking**. "
    "1. **Strict Separation:** Identify and strictly separate all distinct, high-level topics, events, or threads of thought. A topic is distinct if it could be separated into its own paragraph in a book (e.g., 'Work Project Update' vs. 'Personal Event' vs. 'Local News'). "
    "2. **Self-Contained Summary:** For each distinct topic, create a **complete, self-contained summary paragraph**. This summary must be fully understandable without reading the original entry or other sections. This text will be used directly for a search index."
    "3. **Tone Preservation (Optional):** Maintain the original tone (e.g., frustrating, satisfied, stressed) where relevant in the summary."
    "The final output MUST strictly adhere to the provided JSON schema.")'''
    system_prompt = (
    "You are an expert diary analyst and a meticulous data extraction agent. "
    "Your core task is to process a single daily entry and perform **semantic chunking**. "
    "1. **Strict Separation:** Identify and strictly separate all distinct, high-level topics, events, or threads of thought. "
    "2. **Self-Contained Summary:** For each distinct topic, create a **complete, self-contained summary paragraph**. This summary must be fully understandable without reading the original entry or other sections. This text will be used directly for a search index. "
    "**CRITICAL INSTRUCTION: When summarizing, you MUST preserve all proper nouns, including names of people, places, projects, and products exactly as they appear in the original text. Do not substitute them with generic terms (e.g., 'Tom' must remain 'Tom', not 'colleague').** "
    "3. **Keywords:** Ensure the keywords list is useful for search, including the preserved proper nouns. "
    "The final output MUST strictly adhere to the provided JSON schema."
)
    system_prompt = (
    "You are an expert diary analyst and a multilingual data extraction agent. "
    "Your core task is to process a single daily entry and perform **semantic chunking**. "
    "The input may be in English, German, or French, and may contain spelling or grammatical errors. "
    "\n\n"
    "1. **Language & Correction:** You must process the text regardless of the language. "
    "Silently correct any spelling mistakes or typos to ensure high-quality search indexing. "
    "**Crucially, the output (summary and keywords) must be in English**, regardless of the input language, to ensure a uniform search index. "
    "\n"
    "2. **Strict Separation:** Identify and strictly separate all distinct, high-level topics, events, or threads of thought. "
    "\n"
    "3. **Self-Contained Summary:** For each topic, create a **complete, self-contained summary paragraph in English**. "
    "This text must be fully understandable without the original entry. "
    "\n"
    "4. **Proper Nouns:** You MUST preserve all proper nouns (names of people like 'Tom', places, projects) exactly as they appear. "
    "\n"
    "5. **Keywords:** Provide 3-5 English keywords, including any preserved proper nouns." "The final output MUST strictly adhere to the provided JSON schema."
)
    system_prompt = (
    "You are an expert diary analyst and multilingual data extraction agent. "
    "Process each daily entry by identifying distinct semantic chunks (topics, events, reflections). "
    "Input may be in English, German, or French, with possible spelling errors. "
    "\n\n"
    "**Requirements:**\n"
    "1. **Language Processing & Error Correction:** Accept any input language (EN/DE/FR). "
    "**All output (summaries, keywords) MUST be in English** for uniform indexing.\n"
    "   - Silently correct spelling/grammar errors, including common typo patterns:\n"
    "     • Letter transpositions (mavrin → Marvin, hte → the)\n"
    "     • Missing spaces (meetingtomorrow → meeting tomorrow)\n"
    "     • Repeated letters (goood → good, Tomm → Tom)\n"
    "     • Common substitutions (recieve → receive)\n"
    "   - **CRITICAL:** For proper nouns, use context to determine correct spelling. "
    "If 'Marvin' appears correctly 5 times and 'mavrin' once, standardize to 'Marvin'.\n"
    "\n"
    "2. **Semantic Chunking:** Identify ALL distinct topics/events/thoughts. "
    "Each chunk = one searchable concept. Separate unrelated ideas even if in the same paragraph.\n"
    "\n"
    "3. **Self-Contained Summaries (2-4 sentences):** Each summary must:\n"
    "   - Stand alone without requiring the original entry\n"
    "   - Preserve temporal context with explicit time markers:\n"
    "     • Past: 'Yesterday discussed...', 'Last week completed...'\n"
    "     • Present: 'Currently working on...', 'Today experienced...'\n"
    "     • Future: 'Plans to...', 'Will meet... next Tuesday'\n"
    "     • Ongoing: 'Has been regularly...', 'Continues to...'\n"
    "   - Capture specific details EXACTLY (numbers, measurements, technical terms, quotes)\n"
    "   - Maintain emotional tone when present (frustrated, excited, anxious, satisfied)\n"
    "   - Include WHO, WHAT, WHERE, WHEN, WHY/HOW when available\n"
    "\n"
    "4. **Proper Noun Preservation & Standardization:**\n"
    "   - Keep ALL proper nouns exactly as written: Person names (Tom, Dr. Schmidt), "
    "Places (Paris, Café Central), Projects/products (Project Alpha, USS Enterprise), Specific items (book titles, ship names)\n"
    "   - **Consistency Rule:** If the same entity appears with different spellings "
    "(Marvin/mavrin, Café Central/Cafe Central), standardize to the most complete/correct form. "
    "Prioritize: (a) capitalized versions, (b) versions with diacritics/accents, (c) most frequent spelling.\n"
    "\n"
    "5. **Keywords (4-6 per chunk):**\n"
    "   - ALWAYS include preserved proper nouns (people, places, projects)\n"
    "   - Add semantic concepts: broad categories (work, health, relationships) + "
    "specific topics (ship specifications, budget discussion, career anxiety)\n"
    "   - Use natural search terms (how would someone query this in 1-3 words?)\n"
    "   - Example: ['Marvin', 'budget meeting', 'Q4 projections', 'work', 'timeline concerns']\n"
    "\n"
    "6. **Factual Accuracy & Specificity:**\n"
    "   - For factual content (statistics, technical specs, measurements, dates, costs), "
    "preserve EXACT numbers and units in the summary\n"
    "   - For direct quotes or unique phrasings, preserve the wording\n"
    "   - For ambiguous pronouns (he, she, they), replace with the actual person's name if known from context\n"
    "   - If unsure about a correction (is 'mavrin' a typo or an intentional spelling?), "
    "default to the version that appears most frequently across entries\n"
    "\n"
    "Output must conform to the provided JSON schema."
)
    

 
    # 2. Construct the User Message
    user_content = (
        "Here is the daily diary entry text:\n"
        f"---DIARY ENTRY---\n{text_content}\n---END ENTRY---\n"
        "Please analyze the entry and generate the structured JSON output as a list of sections."
    )

    # 3. Call Ollama
    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            format=output_schema,
            options={'temperature': 0.1}, # Slightly higher temp can sometimes help break bad cycles, but keep it low
            keep_alive=0
        )
        
        json_output = response['message']['content']
        
        # 4. Parse and Validate the JSON Response
        sections_model = SectionsList.model_validate_json(json_output)
        
        # Convert the Pydantic model back to a list of standard dictionaries
        return [section.model_dump() for section in sections_model.sections]
    
    except Exception as e:
        # Log the error and return an empty list for this entry
        print(f"Error processing text content: '{text_content[:50]}...'. Error: {e}")
        return []

# --- 2. Iteration and Data Aggregation ---

def process_dataframe(df: pd.DataFrame, content_column: str = 'content_cleaned', date_column: str = 'date') -> pd.DataFrame:
    all_rag_chunks = []
    print(f"Starting semantic chunking for {len(df)} entries...")
    
    for index, row in tqdm(df.iterrows(), total=len(df)):
        original_date = row[date_column]
        original_text = row[content_column] # The raw journal entry
        
        chunks = process_entry_for_rag(original_text)
        
        for chunk in chunks:
            chunk['original_date'] = original_date
            chunk['original_entry_index'] = index
            # --- FIX: Pass the raw content into the chunk so retrieval can find it later ---
            chunk['content'] = original_text 
            all_rag_chunks.append(chunk)

    return pd.DataFrame(all_rag_chunks)
# --- 3. Example Usage with a Dummy DataFrame ---

# Create a sample DataFrame that mimics your structure
data = {
    'date': ['2025-12-14', '2025-12-15', '2025-12-16'],
    'content': [
        "Woke up early, watched the news. The local city council voted down the new park proposal, very frustrating. "
        "Later, I spent three hours completely rewriting the documentation for the 'Authentication' module at work, "
        "it was complex but I feel satisfied with the clarity now. Dinner was chicken and rice.",
        
        "Great day! Finished the Auth module documentation early. Decided to go hiking in the afternoon, "
        "the weather was perfect. Met up with Tom and we discussed the budget for the new app. "
        "He seemed hesitant about the timeline. Got a call from my mother, she's doing fine.",
        
        "The cat was sick this morning, had to take her to the vet (cost $150). Very stressful start. "
        "Had a team meeting, things are moving slowly on the data migration project. Tried a new coffee shop, "
        "the latte was excellent."
    ]
}


# Run the processing pipeline
rag_chunks_df = process_dataframe(df, content_column='content', date_column='date')

print("\n\n--- Final RAG Chunks DataFrame ---")
print(rag_chunks_df.head(10))


# The key column for your next step (Embedding) is 'topic_summary'
print("\nExample of a ready-to-embed chunk:")
print(rag_chunks_df.iloc[0]['topic_summary'])

In [ ]:

class FrenchAssessment(BaseModel):
    comprehensibility_score: int = Field(..., description="1-10 score.")
    estimated_level: str = Field(..., description="CEFR level (e.g., A2).")
    grammar_feedback: str = Field(..., description="Summary of grammar performance.")
    mistakes: List[Dict[str, str]] = Field(..., description="List of 'original' vs 'corrected' terms.")
    suggestions: List[str] = Field(..., description="Tips for improvement.")

def assess_french_skills(text_content: str, model_name: str = 'deepseek-r1:7b') -> Dict[str, Any]:
    """Analyzes French proficiency while ignoring missing accents."""
    
    system_prompt = (
        "You are a French language tutor. Analyze the entry for language quality.\n"
        "1. **ACCENT LENIENCY:** The user uses an English keyboard. Do NOT penalize for missing accents (e.g., 'mange' instead of 'mangé'). "
        "Simply note them in the corrections.\n"
        "2. **FOCUS:** Identify actual grammatical errors (conjugation, gender, syntax) and better word choices.\n"
        "3. **OUTPUT:** Must be in English, following the JSON schema."
    )

    try:
        response = ollama.chat(
            model=model_name,
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": text_content}],
            format=FrenchAssessment.model_json_schema(),
            options={'temperature': 0.1, 'num_predict': 1000}
        )
        return json.loads(response['message']['content'])
    except Exception:
        return {}
def process_dataframe_with_assessment(df: pd.DataFrame, content_col: str = 'content', date_col: str = 'date') -> pd.DataFrame:
    all_rows = []
    print(f"Analyzing {len(df)} entries for RAG and Language Skills...")
    
    for index, row in tqdm(df.iterrows(), total=len(df)):
        text = row[content_col]
        date = row[date_col]
        
        # 1. Get Semantic Chunks for RAG
        chunks = process_entry_for_rag(text)
        
        # 2. Get French Assessment for the whole day
        assessment = assess_french_skills(text)
        
        # 3. Combine data
        for chunk in chunks:
            chunk['original_date'] = date
            chunk['full_content'] = text
            # Inject assessment data into the row
            chunk['french_score'] = assessment.get('comprehensibility_score')
            chunk['french_level'] = assessment.get('estimated_level')
            chunk['grammar_notes'] = assessment.get('grammar_feedback')
            chunk['suggested_corrections'] = assessment.get('mistakes')
            
            all_rows.append(chunk)

    return pd.DataFrame(all_rows)

# Run the pipeline
rag_and_stats_df = process_dataframe_with_assessment(df)

In [ ]:
class FrenchAssessment(BaseModel):
    estimated_level: str = Field(..., description="Estimated CEFR level (e.g., A1, A2, B1, B2, C1).")
    comprehensibility_score: int = Field(..., description="Score from 1-10 on how easy it is to understand.")
    detected_mistakes: List[Dict[str, str]] = Field(..., description="List of 'original' vs 'correction' for grammar/vocab.")
    keyboard_handling: str = Field(..., description="A note confirming accents were ignored for the score due to keyboard limits.")
    improvements: List[str] = Field(..., description="Specific suggestions to move to the next CEFR level.")

def assess_french_skills(text_content: str, model_name: str = 'deepseek-r1:7b') -> Dict[str, Any]:
    system_prompt = (
        "You are an expert French language tutor. Analyze the provided text for language proficiency.\n\n"
        "1. **LEVEL ASSESSMENT:** Assign a CEFR level (A1-C2). "
        "B1 implies the ability to describe events and goals; B2 implies technical accuracy and nuanced expression.\n"
        "2. **ACCENT LENIENCY:** The user is using an English keyboard. **DO NOT** penalize the score or level for missing "
        "accents (e.g., 'j'ai fini' vs 'j'ai fini'). Simply provide the correct version in the mistakes section.\n"
        "3. **CORRECTIONS:** Identify errors in verb conjugation, gender agreement, and word choice.\n"
        "4. **IMPROVEMENTS:** Suggest 3 specific ways to make the writing more natural or advanced (e.g., using 'dont' instead of 'que').\n"
        "The output must be strictly valid JSON."
    )

    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Assess this French entry:\n{text_content}"}
            ],
            format=FrenchAssessment.model_json_schema(),
            options={'temperature': 0.1, 'num_predict': 800} # num_predict prevents hanging/loops
        )
        return json.loads(response['message']['content'])
    except Exception as e:
        print(f"Assessment failed: {e}")
        return {"estimated_level": "N/A", "improvements": []}
def process_dataframe_with_tutor(df: pd.DataFrame):
    all_results = []
    # Set pandas display to infinity for checking results
    pd.set_option('display.max_colwidth', None)
    
    for index, row in tqdm(df.iterrows(), total=len(df)):
        original_text = row['content']
        
        # 1. RAG Chunks (Existing Logic)
        chunks = process_entry_for_rag(original_text)
        
        # 2. French Assessment (New Logic)
        # We only run this if there is enough text to analyze
        assessment = assess_french_skills(original_text)
        
        for chunk in chunks:
            chunk['date'] = row['date']
            chunk['french_level'] = assessment.get('estimated_level')
            chunk['corrections'] = assessment.get('detected_mistakes')
            chunk['how_to_improve'] = assessment.get('improvements')
            all_results.append(chunk)
            
    return pd.DataFrame(all_results)
fr_assessment = process_dataframe_with_tutor(df)

In [ ]:
pd.set_option('display.max_colwidth', None)
print(rag_and_stats_df.columns)
rag_and_stats_df['french_level']
print(fr_assessment.columns)
fr_assessment['french_level']


In [ ]:

#print(old_system.head(20))
#print(old_system.iloc[0]['topic_summary'])
print(old_system['topic_summary'])
print(old_system2['topic_summary'])
#print(old_system.iloc[0]['content'])

In [ ]:
print(rag_chunks_df['topic_summary'])

In [ ]:
import torch
import gc

# 1. Clear out Python's references
gc.collect()

# 2. Clear the CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")
else:
    print("No CUDA device found.")

In [ ]:
import torch
import gc
import ctypes

def intensive_gpu_clean():
    # 1. Clear out Python's references
    gc.collect()
    
    # 2. Clear the CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
        # 3. Use ctypes to talk to the CUDA driver directly (Advanced)
        # This attempts to free the memory that 'empty_cache' misses
        try:
            torch.cuda.ipc_collect()
        except:
            pass
            
    print("Intensive GPU clean complete.")

intensive_gpu_clean()

In [ ]:
def get_relevant_context(query, df, embed_model, k=3):
    """
    Pure function: Uses Gemma 300M to find the best summaries,
    but returns the ORIGINAL raw content for the LLM.
    """
    # 1. Embed the query with the 'Retrieval-query' task
    query_vec = embed_model.encode([query], prompt_name="Retrieval-query")
    
    # 2. Extract the matrix of pre-computed embeddings
    # (Assuming you ran prepare_database first)
    matrix = np.array(df['embedding'].tolist())
    
    # 3. Calculate similarity
    similarities = cosine_similarity(query_vec, matrix).flatten()
    
    # 4. Get the top K entries
    top_indices = similarities.argsort()[-k:][::-1]
    relevant_rows = df.iloc[top_indices]
    
    # 5. Combine original content into a single string
    context_list = []
    for _, row in relevant_rows.iterrows():
        entry = f"--- Date: {row['original_date']} ---\n{row['content']}" # Using original 'content'
        context_list.append(entry)
        
    return "\n\n".join(context_list)
def ask_journal(question, df, embed_model):
    # Retrieve the raw, original journal text
    context = get_relevant_context(question, df, embed_model)
    
    prompt = f"""
    You are a personal diary assistant. Answer the question based ONLY on the 
    provided journal entries. If the answer isn't there, say you don't know.

    RELEVANT ENTRIES:
    {context}

    QUESTION: {question}
    """
    
    # Use keep_alive=0 to ensure the GPU is cleared after the answer
    response = ollama.generate(model='deepseek-r1:7b', prompt=prompt, keep_alive=0)
    return response['response']
# Load Gemma 300M
embed_model = SentenceTransformer("google/embeddinggemma-300m")

# Add embeddings to your DataFrame (using the 'Retrieval-document' task)
embeddings = embed_model.encode(
    rag_chunks_df['topic_summary'].tolist(), 
    prompt_name="Retrieval-document"
)
rag_chunks_df['embedding'] = list(embeddings)
answer = ask_journal("Which ship facts did I write down ? write the corresponding sentences concicely as they were noted! Some of the entries are in German, some in French some in English check for every occurence in these language in ship related facts", rag_chunks_df, embed_model)
print(answer)

In [ ]:
rag_chunks_df.columns

In [ ]:

#print(rag_chunks_df.iloc[2:3]['topic_summary'])
print(rag_chunks_df.head(3))
print(df[df['date']=='15-05-2025']['content'])
# idea for search create embedding for keywords
# make a simple keyword search 
# search for embeddigns of paragraphs 
# get the indexes of the ones that are close 
# feed these as context into the llm 
# generate the answer


In [ ]:
import pandas as pd

query = "SELECT * FROM daily_notes;"
df_retrieval = pd.read_sql(query, engine)

df_retrieval.shape

In [ ]:

warnings.filterwarnings('ignore')

class ArticleRAG:
    def __init__(self, df: pd.DataFrame, content_column: str = 'content', model: str = 'deepseek-r1'):
        """
        Initialize the RAG system with a pandas DataFrame
        
        Args:
            df: DataFrame containing articles
            content_column: Name of the column containing article content
            model: Ollama model name (e.g., 'deepseek-r1', 'deepseek-coder')
        """
        self.df = df.copy().reset_index(drop=True)  # Reset index to ensure continuous indices
        self.content_column = content_column
        self.model = model
        self.vectorizer = None
        self.tfidf_matrix = None
        
        self._build_index()
    
    def _build_index(self):
        """Build TF-IDF index for similarity search"""
        print("Building TF-IDF index...")
        
        # Preprocess text
        texts = self.df[self.content_column].fillna('').astype(str)
        
        # Create TF-IDF vectorizer
        self.vectorizer = TfidfVectorizer(
            stop_words='english',
            max_features=10000,
            ngram_range=(1, 2),
            lowercase=True
        )
        
        # Fit and transform texts
        self.tfidf_matrix = self.vectorizer.fit_transform(texts)
        print(f"Index built with {self.tfidf_matrix.shape[0]} articles and {self.tfidf_matrix.shape[1]} features")
    
    def _semantic_search(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """
        Perform semantic search using TF-IDF similarity
        
        Args:
            query: Search query
            top_k: Number of top results to return
            
        Returns:
            List of (dataframe_index, score) tuples
        """
        # Transform query
        query_vector = self.vectorizer.transform([query.lower()])
        
        # Calculate similarities
        similarities = cosine_similarity(query_vector, self.tfidf_matrix).flatten()
        
        # Get top-k results (matrix indices)
        top_matrix_indices = np.argsort(similarities)[::-1][:top_k]
        
        # Convert matrix indices to DataFrame indices
        df_indices = self.df.index.tolist()
        results = []
        
        for matrix_idx in top_matrix_indices:
            if similarities[matrix_idx] > 0 and matrix_idx < len(df_indices):
                df_idx = df_indices[matrix_idx]
                results.append((df_idx, similarities[matrix_idx]))
        
        return results
    
    def _keyword_search(self, query: str, case_sensitive: bool = False) -> List[int]:
        """
        Perform simple keyword search
        
        Args:
            query: Search query
            case_sensitive: Whether to perform case-sensitive search
            
        Returns:
            List of matching article indices
        """
        if case_sensitive:
            mask = self.df[self.content_column].str.contains(query, na=False, regex=False)
        else:
            mask = self.df[self.content_column].str.contains(query, na=False, case=False, regex=False)
        
        return self.df[mask].index.tolist()
    
    def search_articles(self, query: str, method: str = 'hybrid', top_k: int = 5) -> pd.DataFrame:
        """
        Search for articles based on query
        
        Args:
            query: Search query
            method: 'semantic', 'keyword', or 'hybrid'
            top_k: Number of results to return
            
        Returns:
            DataFrame with matching articles and scores
        """
        if method == 'keyword':
            indices = self._keyword_search(query)
            results_df = self.df.loc[indices].copy()
            results_df['relevance_score'] = 1.0  # All keyword matches get score 1.0
            
        elif method == 'semantic':
            search_results = self._semantic_search(query, top_k)
            if not search_results:
                return pd.DataFrame()
            
            indices, scores = zip(*search_results)
            results_df = self.df.loc[list(indices)].copy()
            results_df['relevance_score'] = scores
            
        else:  # hybrid
            # Combine keyword and semantic search
            keyword_indices = set(self._keyword_search(query))
            semantic_results = self._semantic_search(query, top_k * 2)
            
            # Create combined results
            all_results = {}
            
            # Add keyword matches with high scores
            for idx in keyword_indices:
                all_results[idx] = 0.8  # High score for exact matches
            
            # Add semantic matches
            for idx, score in semantic_results:
                if idx in all_results:
                    all_results[idx] = max(all_results[idx], score + 0.2)  # Boost if also keyword match
                else:
                    all_results[idx] = score
            
            # Sort by score and take top_k
            sorted_results = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:top_k]
            
            if not sorted_results:
                return pd.DataFrame()
            
            indices, scores = zip(*sorted_results)
            results_df = self.df.loc[list(indices)].copy()
            results_df['relevance_score'] = scores
        
        return results_df.sort_values('relevance_score', ascending=False)
    
    def query_with_llm(self, question: str, context_articles: int = 3) -> Dict:
        """
        Use LLM to answer questions based on retrieved articles
        
        Args:
            question: Question to answer
            context_articles: Number of top articles to use as context
            
        Returns:
            Dictionary with answer, sources, and metadata
        """
        # Search for relevant articles
        relevant_articles = self.search_articles(question, method='hybrid', top_k=context_articles)
        
        if relevant_articles.empty:
            return {
                'answer': 'No relevant articles found for your question.',
                'sources': [],
                'confidence': 0.0
            }
        
        # Prepare context
        context = ""
        sources = []
        
        for idx, row in relevant_articles.iterrows():
            article_text = row[self.content_column][:1000]  # Limit context length
            context += f"\nArticle {idx}:\n{article_text}\n"
            sources.append({
                'index': idx,
                'relevance_score': row['relevance_score'],
                'preview': article_text[:200] + "..."
            })
        
        # Create prompt
        prompt = f"""Based on the following articles, please answer the question: "{question}"

Articles:
{context}

Please provide a comprehensive answer based on the information in these articles. If the articles don't contain enough information to answer the question, please state that clearly.

Answer:"""
        
        try:
            # Call Ollama
            response = ollama.chat(
                model=self.model,
                messages=[{'role': 'user', 'content': prompt}]
            )
            
            answer = response['message']['content']
            
            return {
                'answer': answer,
                'sources': sources,
                'confidence': relevant_articles['relevance_score'].mean()
            }
            
        except Exception as e:
            return {
                'answer': f'Error calling LLM: {str(e)}',
                'sources': sources,
                'confidence': 0.0
            }

# Example usage and demo
def demo_rag_system():
    """Demo function showing how to use the RAG system"""
    
    # Create sample data (replace with your actual dataframe)
    sample_data = {
        'content': [
            "Rick and Morty is an animated science fiction sitcom that follows the adventures of an alcoholic scientist and his good-hearted but easily influenced grandson.",
            "The show Rick and Morty has gained massive popularity for its dark humor and complex storylines involving parallel dimensions.",
            "Breaking Bad is considered one of the greatest TV shows of all time, following Walter White's transformation.",
            "Game of Thrones captivated audiences worldwide with its complex political intrigue and fantasy elements.",
            "Rick Sanchez, the genius scientist from Rick and Morty, travels through different dimensions with his grandson Morty.",
            "The Office is a mockumentary sitcom that depicts the everyday work lives of office employees."
        ]
    }
    
   
    
    # Initialize RAG system
    rag = ArticleRAG(df, content_column='content', model='deepseek-r1:7b')
    
    print("=== RAG System Demo ===\n")
    
    # Example 1: Simple search
    print("1. Simple article search:")
    results = rag.search_articles("rick and morty", method='hybrid', top_k=3)
    print(f"Found {len(results)} articles mentioning 'rick and morty':")
    for idx, row in results.iterrows():
        print(f"  - Article {idx} (Score: {row['relevance_score']:.3f})")
        print(f"    Preview: {row['content'][:100]}...")
    
    print("\n" + "="*50 + "\n")
    
    # Example 2: LLM-powered query
    print("2. LLM-powered question answering:")
    response = rag.query_with_llm("What is the current opinion on rick and morty. usually only very few sentences are relevant the rest of the text is not really related to the question", context_articles=2)
    print(f"Question: What is Rick and Morty about?")
    print(f"Answer: {response['answer']}")
    print(f"Confidence: {response['confidence']:.3f}")
    print(f"Sources used: {len(response['sources'])} articles")

if __name__ == "__main__":
    demo_rag_system()

# Usage with your actual dataframe:
"""
# Load your dataframe
df = pd.read_csv('your_articles.csv')  # or however you load your data

# Initialize RAG system
rag = ArticleRAG(df, content_column='content', model='deepseek-r1')

# Search for articles
results = rag.search_articles("rick and morty", top_k=5)
print(results[['content', 'relevance_score']])

# Ask questions
response = rag.query_with_llm("What articles mention Rick and Morty?")
print(response['answer'])
"""

Ideas for the conditional bandit problem:
- Necessary Context
- Indoor/outdoor activity => implict ?
- injury, weather, temperature, time, weekend/not, solo, 